# 01 · sEEG preprocessing (one subject at a time)

Steps, all parameterised in `config/config.yaml`:

1. **Load** the raw recording, clean channel names, drop non-neural channels, find events, resample
2. **Bad channels** – automatic proposal (flat / variance / line noise / HF noise / neighbour correlation) on the *monopolar* signal, merged with manual and previously saved bads, then **interactive review**
3. **Filtering** – high/low-pass + notch
4. **Re-referencing** – bipolar (default), Laplacian, shaft-CAR, CAR or none
5. **Bad temporal intervals** – automatic `BAD_auto` annotations (amplitude, gradient, spike-band envelope), then **interactive review**
6. **Epoching** – event-locked (or block mode) with rejection of epochs overlapping `BAD_*` annotations
7. **Save** epochs (`-epo.fif`), bad channels (`.json`), bad segments (`.csv`), channel QC and a provenance report to `derivatives/preprocessing/sub-<ID>/`

> **Interactive review.** Set `INTERACTIVE = True` to open the MNE browser:
> click a channel name to toggle it bad; press `a` to enter annotation mode and drag to mark a `BAD_*` interval.
> Close the window to continue. The figure needs a GUI backend (`%matplotlib qt`), or install `mne-qt-browser`.

## 0 · Parameters

In [1]:
import os

CONFIG_PATH = os.environ.get("SEEG_CONFIG", "../config/config.yaml")   # or config_demo.yaml
SUBJECT     = os.environ.get("SEEG_SUBJECT", None)   # None -> first subject in the config
INTERACTIVE = os.environ.get("SEEG_INTERACTIVE", "1") == "1"   # open the MNE browser for review

In [2]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))          # make `seeg_wm` importable from notebooks/

get_ipython().run_line_magic("matplotlib", "qt" if INTERACTIVE else "inline")

import matplotlib.pyplot as plt
import mne
import pandas as pd

from seeg_wm import SEEGPreprocessor, load_config
from seeg_wm import channels as chn
from seeg_wm import plotting as splt
from seeg_wm.utils import get_logger, set_seed

cfg = load_config(CONFIG_PATH)
log = get_logger(cfg["project"]["log_level"])
set_seed(cfg["project"]["random_seed"])
mne.set_log_level("WARNING")

SUBJECT = str(SUBJECT or cfg.subjects[0])
print(f"config  : {cfg.path}")
print(f"subject : {SUBJECT}")
print(f"raw file: {cfg.raw_fname(SUBJECT)}")
print(f"outputs : {cfg.subject_dir(SUBJECT)}")

config  : /Users/administrator/__PC_HSE/Results/Nikita Otstavnov/Nikita sEEG/WM_sEEG_project/config/config.yaml
subject : 1
raw file: /Users/administrator/__PC_HSE/Results/Nikita Otstavnov/Nikita sEEG/2011/100726.edf
outputs : /Users/administrator/__PC_HSE/Results/Nikita Otstavnov/Nikita sEEG/WM_sEEG_project/derivatives/preprocessing/sub-1


## 1 · Load raw data and events

In [3]:
pp = SEEGPreprocessor(cfg, SUBJECT)
raw = pp.load()
print(raw)
print("sEEG contacts per shaft:",
      {s: len(c) for s, c in chn.group_by_shaft(pp.seeg_names).items()})

[INFO] Dropping 23 unparsable channel names: ['MKR1+', 'E-Fp1', 'E-Fp2', 'E-F3', 'E-F4', 'E-C3', 'E-C4', 'E-P3', 'E-P4', 'E-F7', 'E-F8', 'E-T3', 'E-T4', 'E-T5', 'E-T6']


RuntimeError: No annotation description matches events.event_id / annotation_map. Found descriptions: []

In [ ]:
raw = mne.io.read_raw_edf('/Users/administrator/__PC_HSE/Results/Nikita Otstavnov/Nikita sEEG/2011/100726.edf', preload=True)

/var/folders/73/99zh84sx6gx2kqgq02_rm3zh0000gp/T/ipykernel_46400/3052648461.py:1: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf('/Users/administrator/__PC_HSE/Results/Nikita Otstavnov/Nikita sEEG/2011/100726.edf', preload=True)


In [ ]:
counts = pd.Series({k: int((pp.events[:, 2] == v).sum()) for k, v in pp.event_id.items()},
                   name="n_events")
display(counts.to_frame())
fig = mne.viz.plot_events(pp.events, sfreq=raw.info["sfreq"], first_samp=raw.first_samp,
                          event_id=pp.event_id, show=False)
fig.set_size_inches(10, 3)

## 2 · Bad channels

Metrics are computed on the monopolar, *un-notched* signal (robust z-scores across contacts):
`flat` (MAD std < threshold), `high_variance`, `line_noise`, `hf_noise` and `low_neighbor_corr`
(median correlation with adjacent contacts on the same shaft). Thresholds: `bad_channels.auto`.

Automatic detection is only a **proposal**. Contacts in the seizure-onset zone or in white matter are
not detected automatically. Add them under `bad_channels.manual` or mark them in the browser.

In [ ]:
metrics = pp.detect_bad_channels()
display(metrics[metrics["auto_bad"]].round(2))
fig = splt.plot_channel_qc(metrics, cfg)

In [ ]:
# Monopolar PSD per contact (bad candidates in red)
spec = raw.compute_psd(fmax=min(250, raw.info["sfreq"] / 2), picks="seeg", exclude=[],
                       tmax=min(raw.times[-1], 120))
raw.info["bads"] = list(pp.auto_bads)
fig = spec.plot(picks="seeg", exclude=[], show=False)
raw.info["bads"] = []

In [ ]:
# Merge automatic + manual (config) + previously saved bads; add ad-hoc ones in `extra`
bads = pp.mark_bad_channels(extra=[])
print("bad channels:", bads)

In [ ]:
# Interactive review: click channel names to (un)mark them as bad, then close the window
if INTERACTIVE:
    pp.plot_raw(block=True)
print("final bad channels:", pp.raw.info["bads"])

## 3 · Filtering

In [ ]:
pp.filter()

## 4 · Re-referencing

`bipolar` builds consecutive pairs along each shaft (bad contacts excluded; `max_gap` controls
bridging over a bad contact). `laplacian` subtracts the mean of the two neighbours, `shaft_car` the
shaft average, `car` the average of all good contacts.

In [ ]:
raw = pp.rereference()
regions = cfg.load_regions().get(SUBJECT, {})
region_of = chn.assign_regions(raw.ch_names, regions,
                               unassigned=cfg["stats"]["per_region"]["unassigned_label"])
ch_table = pd.DataFrame({"channel": raw.ch_names,
                         "region": [region_of[c] for c in raw.ch_names],
                         "hemisphere": [chn.hemisphere(c) for c in raw.ch_names]})
print(f"{len(raw.ch_names)} channels after {pp.reref_method} re-referencing")
display(ch_table.groupby(["region"]).size().rename("n_channels").to_frame())

## 5 · Bad temporal intervals

If annotations were saved in a previous session (and `bad_segments.use_saved: true`), they are
re-loaded. Otherwise they are detected automatically. All `BAD_*` annotations are excluded from epochs.

In [ ]:
n_loaded = pp.load_saved_bad_segments() if cfg["bad_segments"]["use_saved"] else 0
if not n_loaded and cfg["bad_segments"]["auto"]["enabled"]:
    pp.detect_bad_segments()
display(pp.bad_segments_summary())

In [ ]:
# Interactive review: press "a" to annotate (use a description starting with BAD_),
# drag to add, right-click an annotation to remove it. Close the window to continue.
if INTERACTIVE:
    pp.plot_raw(block=True)
display(pp.bad_segments_summary())

## 6 · Epoching

In [ ]:
epochs = pp.make_epochs()
print(epochs)
fig = epochs.plot_drop_log(show=False)

In [ ]:
# Quick sanity check: epoch-averaged PSD per condition family
fig, ax = plt.subplots(figsize=(7, 4))
for cond in sorted({k.split("/")[0] for k in epochs.event_id}):
    sp = epochs[cond].compute_psd(fmin=cfg["spectral"]["fmin"], fmax=cfg["spectral"]["fmax"],
                                  tmin=cfg["spectral"]["tmin"], tmax=cfg["spectral"]["tmax"])
    ax.semilogy(sp.freqs, sp.get_data().mean(axis=(0, 1)), label=cond)
ax.set(xlabel="Frequency (Hz)", ylabel="PSD (V²/Hz)", title=f"sub-{SUBJECT}")
ax.legend();

## 7 · Save

In [ ]:
outputs = pp.save()
pd.Series(outputs, name="path").to_frame()

## 8 · (Optional) batch-process the remaining subjects non-interactively

This uses the same steps and re-uses any bad channels or segments saved during interactive review.

In [ ]:
RUN_BATCH = False
if RUN_BATCH:
    for s in cfg.subjects:
        if s == SUBJECT:
            continue
        try:
            SEEGPreprocessor(cfg, s).run(interactive=False)
        except Exception as err:
            print(f"sub-{s}: FAILED – {err}")